# Gradient interpretability: the murano gradient toolkit

Every other capture step reads what a model *represents*. This tutorial reads
what training would *move*, and walks the whole gradient toolkit on one tiny
model you can run on a laptop. It records the gradient of a frozen checkpoint,
builds the routing operator, a map of which residual-stream directions at a
late layer feed which coordinates at an earlier layer through the model's own
backward transport, uses that map as a fine-tune lens to tell reuse from
rewiring, and finally fits a gradient sparse autoencoder whose features name
what a fine-tune recruited.

**Key questions**

- What does the gradient of a frozen model look like at a residual layer, and
  how do we record it without an optimizer step?
- How similar is the credit routing of a base model and a fine-tune of it, and
  what does chance similarity look like?
- Can the routing map tell a fine-tune that reused the base's paths from one
  that rewired them, apart from how much each moved the loss?
- Which gradient features does a fine-tune introduce, turn up, or suppress, and
  what do they push the model toward?

**Structure**

1. A base and three updates of it
2. Record gradients on a shared rollout corpus
3. The routing operator and its overlap
4. The gradient-off control
5. The fine-tune lens: reuse versus rewire
6. A gradient dictionary
7. The feature census
8. What a recruited feature promotes

**Model and data.** A tiny two-layer, 32-dimensional Llama built from scratch
in this notebook (random weights, word-level tokenizer), plus three updates of
it: a light fine-tune (reuse), a heavier fine-tune on a different rule
(rewire), and a same-size sign-flip of the light update (the inert control).
The "rollouts" are fixed random token sequences teacher-forced through every
checkpoint, so any difference is in the weights alone. Everything runs on CPU
in under a minute and nothing is downloaded.

**Requirements.** The core install is enough, no extras: `pip install
murano-interp`.

In [1]:
from pathlib import Path

import torch

from murano import (
    MuranoModel,
    Pipeline,
    RolloutBatch,
    GFCOperator,
    GSAE,
    keys,
    permutation_floor,
    pairing_overlap,
)
from murano.steps import LoadRollouts, RecordGradients
from murano.steps.gsae import (
    normalized_gradient_inputs,
    firing_rates,
    classify_features,
    promoted_tokens,
    CENSUS_CLASSES,
)

OUTPUT_DIR = "murano_outputs/gradient_interpretability"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
torch.manual_seed(0)

## 1. A base and three updates of it

We build one tiny random Llama as the base, then make three checkpoints from
it. The light fine-tune learns "copy the token two positions back", a rule the
base can already almost express, so it has little reason to rewire. The heavy
fine-tune learns a different, harder rule (map each symbol to a fixed partner)
for more steps, so it does. The inert arm takes the light fine-tune's weight
change, flips its sign per coordinate, and adds it back to the base: the same
support and per-coordinate size as a real update, in a direction unrelated to
what training did.

In [2]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from transformers import LlamaConfig, LlamaForCausalLM, PreTrainedTokenizerFast

# Four specials plus a 20-symbol alphabet t0..t19.
SPECIALS = {"<pad>": 0, "<s>": 1, "</s>": 2, "<unk>": 3}
VOCAB = dict(SPECIALS)
VOCAB.update({f"t{i}": len(SPECIALS) + i for i in range(20)})
FIRST = len(SPECIALS)


def make_base(path: Path) -> None:
    """Save a tiny randomly initialized Llama and its word-level tokenizer."""
    tok = Tokenizer(WordLevel(vocab=dict(VOCAB), unk_token="<unk>"))
    tok.pre_tokenizer = Whitespace()
    PreTrainedTokenizerFast(
        tokenizer_object=tok,
        unk_token="<unk>",
        pad_token="<pad>",
        bos_token="<s>",
        eos_token="</s>",
        model_max_length=48,
    ).save_pretrained(path)
    cfg = LlamaConfig(
        vocab_size=len(VOCAB),
        hidden_size=32,
        intermediate_size=64,
        num_hidden_layers=2,
        num_attention_heads=4,
        num_key_value_heads=4,
        max_position_embeddings=48,
        pad_token_id=0,
        bos_token_id=1,
        eos_token_id=2,
    )
    LlamaForCausalLM(cfg).save_pretrained(path)


base_dir = Path(OUTPUT_DIR) / "base"
make_base(base_dir)

In [3]:
def train_to_targets(src: Path, dst: Path, inputs, targets, steps: int, lr: float) -> float:
    """Fine-tune the checkpoint at src so inputs predict targets, save to dst.

    The tokenizer files are copied from the base first; only the weights change.
    Returns the final loss so the caller can see the update took.
    """
    make_base(dst)  # writes the tokenizer; weights are overwritten next
    model = LlamaForCausalLM.from_pretrained(src)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    labels = inputs.clone()
    labels[:, :-1] = targets[:, 1:]
    for _ in range(steps):
        loss = model(input_ids=inputs, labels=labels).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    model.save_pretrained(dst)
    return float(loss.detach())


rng = torch.Generator().manual_seed(1)
data = torch.randint(FIRST, len(VOCAB), (32, 16), generator=rng)

# Reuse rule: copy the token two back. Rewire rule: map each symbol to a fixed
# partner (symbol i -> symbol (i * 7 + 3) mod 20), a relabeling the base shares
# no structure with, trained longer.
reuse_targets = torch.roll(data, shifts=2, dims=1)
partner = FIRST + (torch.arange(20) * 7 + 3) % 20
rewire_targets = partner[data - FIRST]

reuse_dir = Path(OUTPUT_DIR) / "reuse"
rewire_dir = Path(OUTPUT_DIR) / "rewire"
print("reuse fine-tune loss:  %.3f" % train_to_targets(base_dir, reuse_dir, data, reuse_targets, 40, 3e-3))
print("rewire fine-tune loss: %.3f" % train_to_targets(base_dir, rewire_dir, data, rewire_targets, 120, 3e-3))

reuse fine-tune loss:  0.523


rewire fine-tune loss: 0.069


In [4]:
# The inert control: take the reuse fine-tune's weight change, flip its sign per
# coordinate, and add it back to the base. Same support and per-coordinate size
# as a real update, but a direction unrelated to what training did.
random_dir = Path(OUTPUT_DIR) / "random"
make_base(random_dir)
base_model = LlamaForCausalLM.from_pretrained(base_dir)
reuse_model = LlamaForCausalLM.from_pretrained(reuse_dir)
flip_gen = torch.Generator().manual_seed(2)
base_state = dict(base_model.named_parameters())
with torch.no_grad():
    for name, reuse_param in reuse_model.named_parameters():
        delta = reuse_param - base_state[name]
        signs = torch.randint(0, 2, delta.shape, generator=flip_gen) * 2.0 - 1.0
        base_state[name].add_(delta * signs)
base_model.save_pretrained(random_dir)
print("random arm written (sign-flipped reuse delta)")

random arm written (sign-flipped reuse delta)


## 2. Record gradients on a shared rollout corpus

A rollout is a prompt plus a completion; here we fix eight random token
sequences with a two-token prompt. `RecordGradients` teacher-forces the frozen
model through them, sums the log-likelihood of the completion tokens inside the
window, and takes one backward pass. The stored `[T, d_model]` tensor is the
gradient at the layer-1 residual stream, the direction training would push that
activation, read without an optimizer step. The store also carries the
teacher-forced NLL, used later to see how much each update moved the loss.

In [5]:
# Eight fixed rollouts of 20 tokens, first two as the prompt; teacher-forced
# through every checkpoint so results differ only in weights.
walks = torch.randint(FIRST, len(VOCAB), (8, 20), generator=torch.Generator().manual_seed(3))
rollouts = RolloutBatch(
    input_ids=[row for row in walks],
    prompt_lengths=[2] * len(walks),
)

base = MuranoModel(str(base_dir), device_map="cpu", dtype=torch.float32)
recorded = Pipeline(
    [
        LoadRollouts(rollouts),
        RecordGradients(base, layer=1, window=16),
    ]
).run()

store = recorded[keys.GRADIENT_RECORD]
print("per-rollout gradient shape:", tuple(store.gradients[0].shape))
print("teacher-forced NLL of the first rollouts:", store.nll[:3])

per-rollout gradient shape: (16, 32)
teacher-forced NLL of the first rollouts: tensor([3.1251, 3.1802, 3.1928])


## 3. The routing operator and its overlap

The routing operator projects each position's gradient onto `k` fixed random
orthonormal directions (drawn once from a seed, shared by every checkpoint) and
transports each direction's component from layer 1 back to layer 0 with a
vector-Jacobian product against the frozen forward map. Row `i` of the
resulting `[k, d_model]` map says which layer-0 coordinates direction `i` feeds.
`pairing_overlap` compares two maps after removing the rank-one loudness
background and reducing to unit-strength singular pairs: 1 means the same
routing, and `permutation_floor` measures what chance agreement looks like
instead of assuming it is 0. Here we read the base against its light fine-tune.

In [6]:
reuse = MuranoModel(str(reuse_dir), device_map="cpu", dtype=torch.float32)

# Same corpus, same seeded basis, same layer pair, only the weights differ.
GF = dict(source_layer=1, target_layer=0, k=16, window=16)
base_map = Pipeline([LoadRollouts(rollouts), GFCOperator(base, **GF)]).run()[keys.GFC_OPERATOR]
reuse_map = Pipeline([LoadRollouts(rollouts), GFCOperator(reuse, **GF)]).run()[keys.GFC_OPERATOR]

print("operator shape:", tuple(base_map.operator.shape))
print("self overlap:", round(pairing_overlap(base_map.operator, base_map.operator), 3))
score = pairing_overlap(base_map.operator, reuse_map.operator)
floor_mean, floor_sd = permutation_floor(base_map.operator, reuse_map.operator)
print(f"base vs reuse overlap: {score:.3f}")
print(f"permutation floor:     {floor_mean:.3f} +- {floor_sd:.3f}")

operator shape: (16, 32)
self overlap: 1.0
base vs reuse overlap: 0.868
permutation floor:     -0.004 +- 0.062


## 4. The gradient-off control

Is the overlap above about the gradient, or about the network it flows through?
`gradient_off=True` sets every read strength to 1, so the operator reads the
frozen transport alone and the gradient contributes nothing. Whatever overlap
survives this switch is carried by the models' paths, not by the gradient read
along them. A fine-tune can therefore lower the full overlap in two ways: by
rewiring transport (the gradient-off overlap drops too) or by redistributing
the gradient over conserved paths (the gradient-off overlap stays high).

In [7]:
GF_OFF = dict(source_layer=1, target_layer=0, k=16, window=16, gradient_off=True)
base_off = Pipeline([LoadRollouts(rollouts), GFCOperator(base, **GF_OFF)]).run()[keys.GFC_OPERATOR]
reuse_off = Pipeline([LoadRollouts(rollouts), GFCOperator(reuse, **GF_OFF)]).run()[keys.GFC_OPERATOR]

off_score = pairing_overlap(base_off.operator, reuse_off.operator)
print(f"gradient-weighted overlap: {score:.3f}")
print(f"gradient-off overlap:      {off_score:.3f}")

gradient-weighted overlap: 0.868
gradient-off overlap:      0.667


## 5. The fine-tune lens: reuse versus rewire

With the gradient-off operator per checkpoint, routing drift, one minus the
gradient-off overlap, reads how far each update moved the credit paths, and the
change in teacher-forced NLL reads how far it moved the loss. The light
fine-tune reuses the base's paths and drifts little though it trained; the heavy
fine-tune on a different rule drifts far because it is rewiring rather than
re-weighting. Two caveats this toy makes honest. The sign-flip arm is only
roughly inert here, because a sign flip on a 32-dimensional model is a large
perturbation, where at real scale the same control barely moves the loss. And
drift-per-nat, the paper's summary that divides drift by loss change, only means
something when the arms move the loss by comparable amounts, which real
checkpoints give and this toy does not. The clean matched-loss version is the
real-model study; the toy shows the mechanism.

In [8]:
rewire = MuranoModel(str(rewire_dir), device_map="cpu", dtype=torch.float32)
random_arm = MuranoModel(str(random_dir), device_map="cpu", dtype=torch.float32)
rewire_off = Pipeline([LoadRollouts(rollouts), GFCOperator(rewire, **GF_OFF)]).run()[keys.GFC_OPERATOR]
random_off = Pipeline([LoadRollouts(rollouts), GFCOperator(random_arm, **GF_OFF)]).run()[keys.GFC_OPERATOR]


def mean_nll(result) -> float:
    kept = result.kept.bool()
    return float(result.nll[kept].mean())


# The headline is routing DRIFT = 1 - overlap: how far the update moved the
# credit paths. dNLL is how much it moved the loss. Reuse drifts little though
# it trained; rewire drifts far.
base_nll = mean_nll(base_off)
arms = {"random": random_off, "reuse": reuse_off, "rewire": rewire_off}

print(f"{'arm':8s} {'overlap':>9s} {'drift':>8s} {'dNLL':>9s}")
for name, arm_off in arms.items():
    overlap = pairing_overlap(base_off.operator, arm_off.operator, rank=8)
    print(f"{name:8s} {overlap:9.3f} {1.0 - overlap:8.3f} {mean_nll(arm_off) - base_nll:9.3f}")

arm        overlap    drift      dNLL
random       0.689    0.311     0.064
reuse        0.667    0.333     0.979
rewire       0.114    0.886     2.641


## 6. A gradient dictionary

The routing operator asks where credit flows. A gradient sparse autoencoder
asks what recurring directions the gradient is made of. We pool the
per-position gradients of the base and the rewiring fine-tune on a larger
shared corpus, RMS-normalize each position so loud and quiet ones weigh alike,
and fit a TopK autoencoder whose decoder rows are a dictionary of gradient
features. `normalized_gradient_inputs` does the pooling; `GSAE.train` fits the
dictionary and reports `fvu`, the fraction of gradient variance it leaves
unexplained. The two `RecordGradients` runs stay at cell level so the reader
sees both captures, not a hidden loop.

In [9]:
# A larger shared corpus so the firing rates below are not read off a handful
# of positions. Same two-token prompt and window as the routing capture.
census_walks = torch.randint(
    FIRST, len(VOCAB), (96, 20), generator=torch.Generator().manual_seed(4)
)
census_rollouts = RolloutBatch(
    input_ids=[row for row in census_walks],
    prompt_lengths=[2] * len(census_walks),
)

base_grads = Pipeline(
    [LoadRollouts(census_rollouts), RecordGradients(base, layer=1, window=16)]
).run()[keys.GRADIENT_RECORD]
rewire_grads = Pipeline(
    [LoadRollouts(census_rollouts), RecordGradients(rewire, layer=1, window=16)]
).run()[keys.GRADIENT_RECORD]

base_inputs = normalized_gradient_inputs(base_grads)
rewire_inputs = normalized_gradient_inputs(rewire_grads)

# One dictionary sees both regimes, so a feature has a well-defined firing rate
# on each checkpoint.
gsae = GSAE.train(torch.cat([base_inputs, rewire_inputs]), m=48, k=4, epochs=6)
print(f"pooled positions per checkpoint: {base_inputs.shape[0]}")
print(f"dictionary: {gsae.m} features, {gsae.k} active per input, fvu {gsae.fvu:.3f}")

pooled positions per checkpoint: 1536
dictionary: 48 features, 4 active per input, fvu 0.702


## 7. The feature census

With one dictionary, we can ask how often each feature fires on the base
checkpoint's gradients versus the rewiring fine-tune's. `firing_rates` counts,
per feature, the fraction of positions where it is among the active `k`.
`classify_features` sorts every feature into four classes by how that rate
shifts: introduced (rare in the base, common after), turned up (already
present, fires much more), suppressed (fires much less), and stable (little
change). On this toy the counts are only illustrative; at scale the paper runs
the same census on a 4096-dimension dictionary over millions of positions.

In [10]:
base_rates = firing_rates(gsae, base_inputs)
rewire_rates = firing_rates(gsae, rewire_inputs)
census = classify_features(base_rates, rewire_rates)

print("feature census (base gradients -> rewired gradients):")
for name in CENSUS_CLASSES:
    print(f"  {name:>11}: {len(census[name])} features")

feature census (base gradients -> rewired gradients):
   introduced: 0 features
    turned_up: 6 features
   suppressed: 1 features
       stable: 41 features


## 8. What a recruited feature promotes

A gradient feature is a direction in the residual stream. Sending it through the
final norm's gain and the unembedding, `W_U (gamma * w_j)`, reads out which
tokens that direction pushes the model toward, the cheap decode the paper uses
to name a recruited feature. We take the feature whose firing rose most from
base to fine-tune and read its top tokens; on a random tiny model these are the
mechanism, not a meaning, but at scale the same call names what a recruited
feature would push a real model to say.

In [11]:
# The feature the rewiring fine-tune leaned on most, decoded on that checkpoint.
feature = int((rewire_rates - base_rates).argmax())
print(f"feature {feature}: base fires {base_rates[feature]:.2f}, "
      f"rewire fires {rewire_rates[feature]:.2f}")
print("it promotes:", promoted_tokens(gsae, rewire, feature, top_k=5))

feature 44: base fires 0.06, rewire fires 0.15
it promotes: ['t19', 't13', 't2', 't5', 't7']


## What next

- [Logit attribution](logit_attribution.ipynb) decomposes a *forward* pass into per-component contributions, the representational complement of the gradient view here.
- [Weight ablation](weight_ablation.ipynb) removes components from the weights; pairing it with the fine-tune lens above asks which components carry the routing an update reuses.